# **Sequence Labeling: Named Entity Recognition (NER)**

In this assignment, we will work on a **sequence labeling** task specifically **Named Entity Recognition (NER)**. Our goal is to train a model to recognize entities such as **persons, organizations, locations, and miscellaneous names** in text.

We will use the **CoNLL-2003 dataset**, a widely used benchmark for NER, and leverage **BERT-based** model for token classification.

Throughout this assignment, you will:
- **Preprocess and tokenize the dataset**
- **Fine-tune a pre-trained BERT model for NER**
- **Train the model using PyTorch and Hugging Face's Transformers library**
- **Evaluate model performance using precision, recall, F1-score, and accuracy**

There are three sections to complete, marked TODO A, TODO B, and TODO C.
Everything else is provided — read it carefully before filling anything in.

## Login to HPC

In [ ]:
ssh username@hpc.itu.dk

### Access a compute node

In [ ]:
srun --pty --gres=gpu:1 --mem=16G --time=1:00:00 bash

### Create conda environment and then activate it

In [ ]:
conda create -n week6_assignment_env python=3
conda activate week6_assignment_env

### Install required libraries

In [ ]:
!pip3 install torch --index-url https://download.pytorch.org/whl/cu126

In [ ]:
!pip install datasets transformers evaluate torch seqeval

## Imports and Setup
Initialize random seeds, import libraries, and define hyperparameters.

In [1]:
# Import required libraries (e.g. transformers, datasets, torch, etc.)
from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForTokenClassification, DataCollatorForTokenClassification, AutoConfig, set_seed)
import torch
from torch.utils.data import DataLoader
import random
import evaluate
from tqdm.auto import tqdm

# Set random seeds
set_seed(42)

# Define hyperparameters (e.g., learning_rate, num_train_epochs, model_name)
dataset_name = "conll2003"
learning_rate = 2e-5
num_train_epochs = 3
model_name = "google-bert/bert-base-cased"

## Load Dataset
Load and preview the dataset.

In [2]:
# Load the dataset using load_dataset
dataset_name = "lhoestq/conll2003"
raw_datasets = load_dataset(dataset_name)

# Inspect the dataset structure
raw_datasets


DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})

### Identify Text and Label Columns
For CoNLL-2003, the text column is typically `tokens` and the label column is `ner_tags`.

In [3]:
# Identify columns in raw_datasets['train'].features
from datasets import ClassLabel, Sequence
text_column_name = "tokens"
label_column_name = "ner_tags"
names = ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']
raw_datasets = raw_datasets.cast_column(
    "ner_tags", 
    Sequence(feature=ClassLabel(names=names))
)
features = raw_datasets["train"].features
label_list = features["ner_tags"].feature.names

Build a label list which will be useful for computing metrics using seqeval.

In [4]:
features = raw_datasets["train"].features
label_list = features["ner_tags"].feature.names


## TODO A: Prepare the Tokenizer and Align Labels
We need a tokenizer that handles subwords, then align each token with the correct label (or `-100` for tokens that don’t align to a label).

a) Fill in the code to ignore label_ids after the line `elif word_id == prev_word_id:`

In [5]:
# Load the tokenizer and model config
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
config = AutoConfig.from_pretrained(model_name, num_labels=len(label_list))

def tokenize_and_align_labels(examples):
    """
    For each example, tokenize the list of tokens and align the original labels 
    to the resulting subwords. Tokens can be split into multiple subwords, so we mark 
    the "extra" subwords with -100 to ignore them in the loss.
    """

    # 1) Tokenize
    # 'is_split_into_words=True' tells the tokenizer each item in the list is already a separate word/token.
    tokenized_inputs = tokenizer(
        examples[text_column_name],  # e.g., examples["tokens"]
        max_length=128,             
        padding=False,              
        truncation=True, 
        is_split_into_words=True
    )

    # 2) Prepare a new "labels" list aligned to the subword tokens
    all_labels = []
    
    # examples[label_column_name] might look like: [0, 0, 1, 2, ...] for each token
    for batch_index, labels in enumerate(examples[label_column_name]):
        # 'word_ids()' returns a list the same length as the subword-tokens,
        # each entry telling you which 'word' or token it came from
        word_ids = tokenized_inputs.word_ids(batch_index=batch_index)

        label_ids = []
        prev_word_id = None
        
        for word_id in word_ids:
            if word_id is None:
                # e.g. special tokens or padding
                label_ids.append(-100)
            elif word_id == prev_word_id:
                # subword token of the same word => ignore 
                label_ids.append(-100)
            else:
                # new subword, so use the label for the original token
                label_ids.append(labels[word_id])
            
            prev_word_id = word_id
        
        all_labels.append(label_ids)

    # 3) Attach the new "labels" to our tokenized inputs
    tokenized_inputs["labels"] = all_labels

    # 4) Return the updated dictionary
    return tokenized_inputs


### Process the Dataset
Apply the tokenization and label alignment function, removing original columns to keep only model inputs.

In [6]:
# Map the tokenize_and_align_labels function to the raw datasets
processed_raw_datasets = raw_datasets.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=raw_datasets["train"].column_names,
    desc="Running tokenizer on dataset"
)

train_dataset = processed_raw_datasets["train"]
eval_dataset = processed_raw_datasets["validation"]

# Inspect a few training samples after tokenization
for index in random.sample(range(len(train_dataset)), 3):
    print(f"Sample {index} of the training set: {train_dataset[index]}")

Sample 10476 of the training set: {'input_ids': [101, 140, 11607, 19747, 2249, 11185, 21669, 13020, 18732, 2162, 9565, 14569, 2346, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': [-100, 3, -100, -100, -100, -100, -100, 0, 5, -100, -100, -100, -100, -100]}
Sample 1824 of the training set: {'input_ids': [101, 9269, 1114, 2733, 117, 1134, 1110, 1412, 1514, 3547, 117, 1138, 1632, 4495, 117, 107, 23209, 1732, 1918, 1163, 119, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': [-100, 0, 0, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, -100, -100, 0, 0, -100]}
Sample 409 of the training set: {'input_ids': [101, 1124, 1896, 131, 107, 1409, 1185, 1141, 1455, 117, 146, 1309, 1533, 1139, 1779, 119, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 

## Define the Model and Data Collator
Set up the token classification model using the config, and create a data collator that can handle token classification tasks.

In [7]:
# Initialize the model with AutoModelForTokenClassification
model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    config=config
)

# Create a data collator
data_collator = DataCollatorForTokenClassification(tokenizer)


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: google-bert/bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly init

## Create DataLoaders
Use the processed dataset and data collator to build PyTorch DataLoader objects.

In [8]:
# Create train and eval dataloaders
train_dataloader = DataLoader(train_dataset, shuffle=True, collate_fn=data_collator, batch_size=8)
eval_dataloader = DataLoader(eval_dataset, collate_fn=data_collator, batch_size=8)


## Optimizer
Initialize an optimizer.

In [ ]:
# Move model to device (CPU/GPU)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print(device)
# Create optimizer (e.g. AdamW)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

cpu


In [28]:
# DO NOT RUN IT. THIS IS FOR AMD GPU as an alternative for the code above
!pip install torch-directml
import torch_directml
device = torch_directml.device() 
model.to(device)

# Create optimizer (e.g. AdamW)
optimizer = torch.optim.SGD(
    model.parameters(), 
    lr=5e-5, 
    momentum=0.9, 
    nesterov=True, 
    weight_decay=0.01
)

## TODO B Training Loop
a) Train for the specified number of epochs, updating the model parameters each step.

b) Print the average loss each epoch

In [29]:
# Implement the training loop
# Set model to train mode
# For each epoch:
#   1. Iterate over batches in train_dataloader
#   2. Zero grads, forward pass, compute loss, backward, step optimizer
#   3. Keep track of total loss

# Example:
# model.train()
# for epoch in range(num_train_epochs):
#     total_loss = 0
#     pbar = tqdm(enumerate(train_dataloader), total=len(train_dataloader), desc=f"Training Epoch {epoch+1}")
#     for step, batch in pbar:
#     Your code here...

import torch
model.train()

for epoch in range(num_train_epochs):
    total_loss = 0
    
    pbar = tqdm(enumerate(train_dataloader), total=len(train_dataloader), desc=f"Epoch {epoch+1}")
    
    for step, batch in pbar:
        #move batch to device, move every tensor(datapoints) in it to GPU/CPU
        batch = {k: v.to(device) for k, v in batch.items()}

        #forward pass
        #tuple-like object for hugging face 
        #loss calcukated automatically cuz we included 'labels' in the batch. 
        outputs = model(**batch)
        loss = outputs.loss

        #backward pass
        loss.backward() 

        #optimizer. wierd shit. I thought you'd have to pass it through the model or something
        #turns out you just need to pass the parameters (as seen in initialization) when making it cause it gets pointers 
        #to the data. So the optimizer has the reference to the models weights already. 
        optimizer.step()  #changes the weights by gradient.
        optimizer.zero_grad() #erases the old gradients after the step() from memory so next iteration you can get new ones. Needed it or gradients will go huge. again wierd shit.

        #track loss
        total_loss += loss.item()
        
        # update progress bar with loss
        pbar.set_postfix({"loss": loss.item()})

    #print average loss
    avg_train_loss = total_loss / len(train_dataloader)
    print(f"Epoch {epoch+1} average training loss: {avg_train_loss:.4f}")

Epoch 1:   0%|          | 0/1756 [00:00<?, ?it/s]

KeyboardInterrupt: 

## TODO C Define Metric Calculation
For example, use `seqeval` to evaluate precision, recall, and F1 on the named entity labels.

In [ ]:
# Load seqeval metric
metric = evaluate.load("seqeval")

a) Define utility function get_labels(predictions, references)

In [ ]:
# TODO: Define utility function get_labels(predictions, references)
# and compute_metrics to return precision, recall, f1, etc.
def get_labels(predictions, references):
    """
    Convert model outputs (logits) and references (label IDs)
    into human-readable label names, ignoring subword tokens.
    
    Args:
        predictions: A PyTorch tensor with shape [batch_size, seq_length],
                     containing the predicted label IDs for each token.
        references:  A PyTorch tensor with the true label IDs for each token.
        
    Returns:
        true_predictions, true_labels:
        - Each is a list of lists of strings.
        - Outer list = batch dimension
        - Inner list = predicted or true labels for each token in that example
        - We skip any token whose label == -100 (these are subword tokens or padding).
    
    Example:
        Suppose label_list = ["O", "B-PER", "I-PER"],
        predictions = [[0, 1, 2], [0, 0, 1]],
        references  = [[0, 1, 2], [0, 0, -100]]
        
        Then,
        true_predictions might be [["O", "B-PER", "I-PER"], ["O", "O"]]
        true_labels      might be [["O", "B-PER", "I-PER"], ["O", "O"]]
    """    
    true_predictions = []
    true_labels = []
    # Your code here...
    return true_predictions, true_labels

In [ ]:
def compute_metrics(preds, refs):
    results = metric.compute(predictions=preds, references=refs)
    return {
        "Precision": results["overall_precision"],
        "Recall": results["overall_recall"],
        "F1": results["overall_f1"],
        "Accuracy": results["overall_accuracy"],
    }

## Evaluation
Switch to eval mode, run inference, and measure performance with the chosen metric.

In [ ]:
# After training, evaluate on the validation set
model.eval()
validation_progress_bar = tqdm(range(len(eval_dataloader)))
all_predictions = []
all_labels = []
for step, batch in enumerate(eval_dataloader):
    batch = {k: v.to(device) for k, v in batch.items()}
    with torch.no_grad():
        outputs = model(**batch)
    predictions = outputs.logits.argmax(dim=-1)
    labels = batch["labels"]
    predicted_labels, true_labels = get_labels(predictions, labels)
    all_predictions.extend(predicted_labels)
    all_labels.extend(true_labels)
    validation_progress_bar.update(1)

validation_metrics = compute_metrics(all_predictions, all_labels)
validation_metrics

## Bonus
**Hyperparameter Tuning**: Try changing learning rates, batch sizes, or number of epochs.